# CMSC 173 &middot; Module 13 &middot; Example 2
## A CNN for the digits

Module 12's dense network read each 8 &times; 8 digit as 64 separate numbers: 0.987 on the test digits. A small CNN reads it as a picture. Then every test digit is moved one pixel.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (PyTorch on the CPU, scikit-learn 1.4.2; a newer version can differ slightly for the trained networks);
change something and see what moves.

**Module 12's dense network**, on module 12's split.

In [ ]:
import numpy as np, torch, torch.nn as nn, warnings
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X / 16, y, test_size=0.3, random_state=173)
dense = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", alpha=1e-4, learning_rate_init=0.001,
                      batch_size=64, max_iter=300, random_state=173).fit(X_tr, y_tr)
print(round(dense.score(X_te, y_te), 3))                       # 0.987

**The CNN.** Two convolution layers, each followed by ReLU and 2 &times; 2 max pooling, then one dense layer of 10 neurons. Count the weights layer by layer.

In [ ]:
torch.manual_seed(173)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 8x8x1 -> 8x8x8 -> 4x4x8
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # -> 4x4x16 -> 2x2x16
    nn.Flatten(), nn.Linear(64, 10))                                           # 64 numbers -> 10 scores
print([sum(p.numel() for p in layer.parameters()) for layer in cnn if list(layer.parameters())])  # [80, 1168, 650]

**Training**: 30 epochs of Adam, batches of 64, the loss of module 12's softmax output.

In [ ]:
img = lambda A: torch.tensor(A, dtype=torch.float32).reshape(-1, 1, 8, 8)   # 64 numbers -> 1 x 8 x 8 picture
Xt, yt = img(X_tr), torch.tensor(y_tr)
opt, loss_fn = torch.optim.Adam(cnn.parameters(), lr=0.01), nn.CrossEntropyLoss()
g = torch.Generator().manual_seed(173)
for epoch in range(30):
    order = torch.randperm(len(Xt), generator=g)
    for i in range(0, len(Xt), 64):
        batch = order[i:i + 64]
        opt.zero_grad(); loss_fn(cnn(Xt[batch]), yt[batch]).backward(); opt.step()
with torch.no_grad():
    cnn_acc = lambda A: (cnn(img(A)).argmax(1).numpy() == y_te).mean()
print(round(cnn_acc(X_te), 3))                                 # 0.976

**Move every test digit one pixel**, right and then left; the empty column fills with zeros.

In [ ]:
def shift(A, dx):
    img8 = A.reshape(-1, 8, 8); out = np.zeros_like(img8)
    if dx > 0: out[:, :, dx:] = img8[:, :, :-dx]
    else:      out[:, :, :dx] = img8[:, :, -dx:]
    return out.reshape(len(A), 64)

for dx in (1, -1):
    print(dx, round(dense.score(shift(X_te, dx), y_te), 3), round(cnn_acc(shift(X_te, dx)), 3))
# 1  0.483 0.624
# -1 0.437 0.6

**What the first layer learned**: its 8 kernels, each a 3 &times; 3 grid of weights.

In [ ]:
import matplotlib.pyplot as plt
w = cnn[0].weight.detach().numpy()[:, 0]
fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for k in range(8):
    ax[k].imshow(w[k], cmap="RdBu_r"); ax[k].set_xticks([]); ax[k].set_yticks([])
plt.show()

**Try it.** Train on the training digits *and* copies moved one pixel each way (stack them with `np.vstack`). How far does the shifted accuracy rise, for each model?